In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated/checkpoints/post_cell_10.pickle

trying: ['file_name_2018', 'file_name_2017']
me:  8
me:  8
trying: ['file_name_2018', 'file_name_2017']
me:  8
me:  8
trying: ['bar_chart_multiple_choice_multiple_selection']
me:  2
trying: ['count_then_return_percent_17']
me:  16
trying: ['create_choropleth_map']
me:  2
trying: ['bar_chart_multiple_choice']
me:  2
trying: ['bar_chart_multiple_years']
me:  2
trying: ['convert_df_of_counts_to_percentages']
me:  4
trying: ['create_dataframe_of_counts']
me:  4
trying: ['count_then_return_percent']
me:  4
trying: ['orig_output']
me:  15
trying: ['add_year_column_to_dataframes']
me:  4
trying: ['glob']
me:  0
trying: ['title_for_y_axis']
me:  20
trying: ['grab_subset_of_data']
me:  4
trying: ['add_year_column_to_dataframes_18']
me:  18
trying: ['load_survey_data']
me:  6
trying: ['warnings']
me:  0
trying: ['combine_subset_of_data_from_multiple_years_18']
me:  18
trying: ['sns']
me:  0
trying: ['responses_df_2022']


me:  18
trying: ['base_dir_2021']
me:  8
trying: ['percentages']
me:  20
trying: ['bar_chart_multiple_years_18']
me:  18
trying: ['file_name_2021']
me:  8
trying: ['title_for_x_axis']
me:  18
trying: ['base_dir_2019']
me:  8
trying: ['question_name_alternate']
me:  18
trying: ['count_then_return_percent_19']
me:  20
trying: ['count_then_return_percent_18']
me:  18
trying: ['base_dir_2018']
me:  8
trying: ['bar_chart_multiple_choice_19']
me:  20
trying: ['country_df_combined']
me:  18
trying: ['question_of_interest']
me:  20
trying: ['subset_of_countries']
me:  18
trying: ['responses_df_2020']


me:  8
trying: ['responses_df_2017']


me:  18
trying: ['responses_df_2021']


me:  8
trying: ['responses_df_2019']


me:  10
trying: ['column_of_interest']
me:  18
trying: ['question_name']
me:  18
trying: ['responses_per_country_df']
me:  14
trying: ['create_choropleth_map_16']
me:  14
trying: ['percentages_per_country_df']
me:  14
trying: ['base_dir_2020']
me:  8
trying: ['base_dir_2017']
me:  8
trying: ['directory']
me:  8
trying: ['base_dir_2022']
me:  8
trying: ['combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions']
me:  6
trying: ['create_dataframe_of_counts_16']
me:  14
trying: ['combine_subset_of_data_from_multiple_years']
me:  6
trying: ['file_name_2022']
me:  8
trying: ['label_for_legend']
me:  14
trying: ['px']
me:  0
trying: ['file_name_2019']
me:  8
trying: ['go']
me:  0
trying: ['file_name_2020']
me:  8
trying: ['orientation_for_chart']
me:  16
trying: ['title_for_chart']
me:  20
trying: ['replace_hyphen_with_en_dash']
me:  10
trying: ['bar_chart_multiple_choice_17']
me:  16
trying: ['responses_df_2018']


me:  10
trying: ['np']
me:  0
trying: ['pd']
me:  0
trying: ['responses_in_order']
me:  16
Declaring variable glob
Declaring variable warnings
Declaring variable sns
Declaring variable px
Declaring variable go
Declaring variable np
Declaring variable pd
Declaring variable bar_chart_multiple_choice_multiple_selection
Declaring variable create_choropleth_map
Declaring variable bar_chart_multiple_choice
Declaring variable bar_chart_multiple_years
Declaring variable convert_df_of_counts_to_percentages
Declaring variable create_dataframe_of_counts
Declaring variable count_then_return_percent
Declaring variable add_year_column_to_dataframes
Declaring variable grab_subset_of_data
Declaring variable load_survey_data
Declaring variable combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions
Declaring variable combine_subset_of_data_from_multiple_years
Declaring variable file_name_2018
Declaring variable file_name_2017
Declaring variable file_name_2018
Declarin

In [4]:
%%RecordEvent
%%time
### cell 11 ###

### cell 11 optimized ###
def bar_chart_multiple_years_20(df, x_column, y_column, title, y_axis_title):
    # write out combined data
    df.to_csv(
        '/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data/'
        + title + '.csv', index=True
    )

def combine_subset_of_data_from_multiple_years_20(question_of_interest, x_axis_title, include_2017=None):
    # map each year to its responses DataFrame
    year_map = {
        '2018': responses_df_2018,
        '2019': responses_df_2019,
        '2020': responses_df_2020,
        '2021': responses_df_2021,
        '2022': responses_df_2022,
    }
    if include_2017:
        year_map = {'2017': responses_df_2017, **year_map}

    # select the column of interest and add a 'year' column for each DataFrame
    df_list = [
        df[[question_of_interest]]
          .rename(columns={question_of_interest: x_axis_title})
          .assign(year=year)
        for year, df in year_map.items()
    ]
    # concatenate all years into one GPU DataFrame
    df_all = pd.concat(df_list)

    # compute counts per (year, category)
    grouped = df_all.groupby(['year', x_axis_title]).size().reset_index(name='count')
    # compute percentage within each year
    grouped['percentage'] = (
        grouped['count']
        * 100
        / grouped.groupby('year')['count'].transform('sum')
    ).round(1)

    # return in the same format: [percentage, year, category]
    return grouped[['percentage', 'year', x_axis_title]]

# parameters and execution
question_of_interest = 'What is your age (# years)?'
column_of_interest = 'percentage'
title_for_chart = 'Age distributions on Kaggle (2018-2022)'
title_for_x_axis = ''
title_for_y_axis = '% of respondents'
# merge the top bins before combining
auto_replace = ['70-79', '80+']
responses_df_2018[question_of_interest].replace(auto_replace, '70+', inplace=True)

# combine and write out
age_df_combined = combine_subset_of_data_from_multiple_years_20(
    question_of_interest,
    title_for_x_axis
)
bar_chart_multiple_years_20(
    age_df_combined,
    title_for_x_axis,
    column_of_interest,
    title_for_chart,
    title_for_y_axis
)


CPU times: user 259 ms, sys: 39.7 ms, total: 298 ms
Wall time: 298 ms


In [5]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high/checkpoints/post_cell_11_try_8.pickle

migration speed (bps): 773402371.5559342
---------------------------
variables to migrate:
orig_output 16
age_df_combined 48
create_dataframe_of_counts_16 144
label_for_legend 65
column_of_interest 59
responses_per_country_df 48
create_choropleth_map_16 144
percentages_per_country_df 48
pd 72
title_for_chart 88
responses_df_2017 48
bar_chart_multiple_choice 144
add_year_column_to_dataframes_18 144
bar_chart_multiple_choice_multiple_selection 144
combine_subset_of_data_from_multiple_years_18 144
base_dir_2020 155
bar_chart_multiple_years 144
base_dir_2017 155
create_choropleth_map 144
directory 170
question_name 90
combine_subset_of_data_from_multiple_years_20 144
base_dir_2022 155
bar_chart_multiple_years_18 144
file_name_2022 81
title_for_x_axis 49
file_name_2018 76
question_name_alternate 70
file_name_2019 78
count_then_return_percent_18 144
file_name_2020 81
country_df_combined 48
convert_df_of_counts_to_percentages 144
subset_of_countries 184
create_dataframe_of_counts 144
count_th

In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['percentages', 'question_of_interest', 'responses_df_2022']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['percentages', 'question_of_interest', 'responses_df_2022']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['percentages', 'question_of_interest', 'responses_df_2022']
Modified dataframes
======= Cell 4 =======
Input variables []
Active variables ['responses_df_2017', 'responses_df_2018', 'responses_df_2020', 'responses_df_2019', 'responses_df_2021', 'responses_df_2022']
Intermediate variables ['directory', 'base_dir_2017', 'file_name_2017', 'file_name_2020', 'base_dir_2018', 'base_dir_2021', 'fil

In [7]:

with open("/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_11_try_8.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[11], f)


In [8]:
opt_output = Out.get(4)

In [9]:
assert False, 'column_of_interest is incorrectly modified in the optimized code.'

AssertionError: column_of_interest is incorrectly modified in the optimized code.